## Day 1: Setup and Foundation

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

# Check if the file exists in your specified Drive path
file_path = '/content/drive/MyDrive/ecommerce_rfm_project/online_retail_II.xlsx'

print("File exists:", os.path.exists(file_path))

File exists: True


## Day 2: Data Ingestion and Inspecting Schemas

In [4]:
import pandas as pd

In [5]:
excel_workbook = pd.ExcelFile(file_path)
excel_workbook.sheet_names


['Year 2009-2010', 'Year 2010-2011']

In [6]:
df_temp = pd.read_excel(excel_workbook, sheet_name='Year 2009-2010')
df_temp.to_csv('year_2009_2010.csv', index=False)

In [7]:
del df_temp

In [8]:
import gc

gc.collect()

525511

In [9]:
df_temp2 = pd.read_excel(excel_workbook, sheet_name='Year 2010-2011')
df_temp2.to_csv('year_2010_2011.csv', index=False)

In [10]:
import gc

del df_temp2
gc.collect()

541960

In [11]:
df_temp1 = pd.read_csv('year_2009_2010.csv')
df_temp2 = pd.read_csv('year_2010_2011.csv')

In [12]:
df_temp1_columns = df_temp1.columns.str.strip().str.lower()
df_temp2_columns = df_temp2.columns.str.strip().str.lower()
print(df_temp1_columns)
print(df_temp2_columns)


Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer id', 'country'],
      dtype='object')
Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer id', 'country'],
      dtype='object')


In [13]:
df_temp1.columns = df_temp1.columns.str.strip().str.lower()
df_temp2.columns = df_temp2.columns.str.strip().str.lower()

In [14]:
print(df_temp1.dtypes)
print(df_temp2.dtypes)

invoice         object
stockcode       object
description     object
quantity         int64
invoicedate     object
price          float64
customer id    float64
country         object
dtype: object
invoice         object
stockcode       object
description     object
quantity         int64
invoicedate     object
price          float64
customer id    float64
country         object
dtype: object


In [15]:
print(df_temp1.isnull().sum())
print(df_temp2.isnull().sum())

invoice             0
stockcode           0
description      2928
quantity            0
invoicedate         0
price               0
customer id    107927
country             0
dtype: int64
invoice             0
stockcode           0
description      1454
quantity            0
invoicedate         0
price               0
customer id    135080
country             0
dtype: int64


In [16]:
total_rows1 = len(df_temp1)
missing_percentage = (df_temp1.isnull().sum() / total_rows1) * 100
print(missing_percentage)

invoice         0.000000
stockcode       0.000000
description     0.557225
quantity        0.000000
invoicedate     0.000000
price           0.000000
customer id    20.539488
country         0.000000
dtype: float64


In [17]:
total_rows2 = len(df_temp2)
missing_percentage = (df_temp2.isnull().sum() / total_rows2) * 100
print(missing_percentage)

invoice         0.000000
stockcode       0.000000
description     0.268310
quantity        0.000000
invoicedate     0.000000
price           0.000000
customer id    24.926648
country         0.000000
dtype: float64


In [18]:
print(df_temp1.describe())
print(df_temp2.describe())

            quantity          price    customer id
count  525461.000000  525461.000000  417534.000000
mean       10.337667       4.688834   15360.645478
std       107.424110     146.126914    1680.811316
min     -9600.000000  -53594.360000   12346.000000
25%         1.000000       1.250000   13983.000000
50%         3.000000       2.100000   15311.000000
75%        10.000000       4.210000   16799.000000
max     19152.000000   25111.090000   18287.000000
            quantity          price    customer id
count  541910.000000  541910.000000  406830.000000
mean        9.552234       4.611138   15287.684160
std       218.080957      96.759765    1713.603074
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


## Day 3: Data Quality Audit and Cleaning Pipeline

In [24]:
combined_df = pd.concat([df_temp1, df_temp2], axis=0)
display(combined_df)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [28]:
filtered_df = combined_df[combined_df['invoice'].str.contains('C', na=False)]
display(filtered_df)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [29]:
(filtered_df['quantity'] <= 0).all()

np.False_

In [30]:
len(combined_df [combined_df['quantity'] < 0])

22950

In [31]:
filtered_df2 = combined_df[~combined_df['invoice'].str.contains('C', na=False)]
display(filtered_df2)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [36]:
negative_quantities = filtered_df2 [filtered_df2['quantity'] < 0]
display(negative_quantities)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom
536910,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom


In [39]:
len(combined_df [(combined_df['price'] <= 0) & (combined_df['quantity']>0)])

2750

In [40]:
filtered_df3 = combined_df [(combined_df['price'] <= 0) & (combined_df['quantity']>0)]
display(filtered_df3)

,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535334,581211,22142,check,14,2011-12-07 18:36:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom


In [43]:
clean_df = combined_df [(combined_df ['quantity'] > 0) & (combined_df ['price'] > 0) & (combined_df ['customer id'].notnull())]
display(clean_df)


,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [56]:
clean_df = clean_df.drop_duplicates()
display(clean_df)


,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [57]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   invoice      779425 non-null  object 
 1   stockcode    779425 non-null  object 
 2   description  779425 non-null  object 
 3   quantity     779425 non-null  int64  
 4   invoicedate  779425 non-null  object 
 5   price        779425 non-null  float64
 6   customer id  779425 non-null  float64
 7   country      779425 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 53.5+ MB


In [71]:
type(clean_df['invoicedate'].iloc[-1])

pandas.core.series.Series

In [73]:
clean_df = combined_df[(combined_df['quantity'] > 0) &
                       (combined_df['price'] > 0) &
                       (combined_df['customer id'].notnull())].copy()

clean_df = clean_df.drop_duplicates()

In [74]:
clean_df['invoicedate'] = pd.to_datetime(clean_df['invoicedate'])

In [75]:
clean_df['customer id'] = clean_df['customer id'].astype('int64')


In [76]:
clean_df['total_amount'] = clean_df['quantity'] * clean_df['price']

In [77]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 541909
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   invoice       779425 non-null  object        
 1   stockcode     779425 non-null  object        
 2   description   779425 non-null  object        
 3   quantity      779425 non-null  int64         
 4   invoicedate   779425 non-null  datetime64[ns]
 5   price         779425 non-null  float64       
 6   customer id   779425 non-null  int64         
 7   country       779425 non-null  object        
 8   total_amount  779425 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 59.5+ MB


In [78]:
clean_df['total_amount'].describe()

,total_amount
count,779425.000000
mean,22.291823
std,227.427075
min,0.001000
25%,4.950000
50%,12.480000
75%,19.800000
max,168469.600000


In [79]:
clean_df.shape


(779425, 9)